In [141]:
import re
import os
import pandas as pd

RAW_FILE = os.path.join("..", "data", "raw", "Cardio_onc_prostate_students.xlsx")
PROCESSED_DIR = os.path.join("..", "data", "processed")
COLUMN_KEY_FILE = os.path.join("..", "data", "column_key.csv")
PROCESSED_FILE = os.path.join(PROCESSED_DIR, "cardio_onc_prostate_02processed.csv")

os.makedirs(PROCESSED_DIR, exist_ok=True)

df = pd.read_excel(RAW_FILE)
print(f"Preprocessed Length: {len(df)}")

Preprocessed Length: 241


## Remove some columns
- don't have unique patient ID
- duplicate PCP column

In [142]:
# Drop rows with missing unique patient ID
if "Unique Patient ID" in df.columns:
    df = df.dropna(subset=["Unique Patient ID"])

print(f"Rows after dropping missing IDs: {len(df)}")

Rows after dropping missing IDs: 239


In [143]:
# ==========================================
# HANDLE DUPLICATE PCP COLUMN
# ==========================================

if "has_pcp_duplicate" in df.columns:
    df = df.drop(columns=["has_pcp_duplicate"])

old_cols = df.columns.tolist()

## Clean Column Names
- manually rename columns with 3+ words

In [144]:
def clean_column_name(col):
    col = col.strip().lower()
    col = re.sub(r"\(.*?\)", "", col)  # remove parentheses
    col = col.split("?")[0]  # remove question mark metadata
    col = col.replace("%", "pct")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.replace(" ", "_")
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

MANUAL_RENAME = {
    "nht_prior_authorization_date": "nht_auth_date",
    "start_date_of_nht": "nht_start_date",
    "start_date_of_adt": "adt_start_date",
    "specific_adt_agent_used": "adt_agent",
    "age_at_nht_start": "age",
    "hx_of_smoking": "hx_smoking",
    "family_hx_of_heart_disease": "family_hx_hd",
    "hx_of_htn_prior_to_nht_start": "hx_htn",
    "hx_of_hyperlipidemia_prior_to_nht_start": "hx_hld",
    "hx_of_high_triglycerides_prior_to_nht_start": "hx_high_tg",
    "bp_medication_within_3_months_prior_to_nht_start": "bp_meds_prior",
    "on_beta_blocker_prior_to_nht": "bb_prior",
    "on_ace_i_or_arb_prior_to_nht": "ace_arb_prior",
    "starting_new_bp_medications_after_nht_start": "bp_meds_post",
    "on_a_statin_prior_to_nht_start": "statin_prior",
    "on_another_cho_lowering_drug_prior_to_nht_start": "other_lipid_prior",
    "starting_new_cholesterol_medications_after_nht_start": "lipid_meds_post",
    "had_lipid_panel_checked_3_months_prior_to_3_months_post_nht_start": "lipid_panel_checked",
    "10_year_ascvd_risk": "ascvd_10yr",
    "10_yr_risk_score_pct": "ascvd_10yr_pct",
    "hx_of_cad_prior_to_nht_start": "hx_cad",
    "date_of_cad": "cad_date",
    "hx_of_mi_or_stent_placement_prior_to_nht_start": "hx_mi_stent",
    "date_of_mi_orstent": "mi_stent_date",
    "hx_of_chf_prior_to_nht_start": "hx_chf",
    "date_of_hf": "hf_date",
    "hx_of_arrhythmia_prior_to_nht_start": "hx_arrhythmia",
    "date_of_arrhythmia": "arrhythmia_date",
    "hx_of_carotid_art._disease_prior_to_nht_start": "hx_carotid",
    "date_of_carotid_art_disease": "carotid_date",
    "hx_of_pvd_pad_prior_to_nht_start": "hx_pad",
    "date_of_pvd_pad": "pad_date",
    "hx_of_cva_prior_to_nht_start": "hx_cva",
    "date_of_cva": "cva_date",
    "hx_of_diabetes_mellitus_type_ii": "hx_dm2",
    "on_oral_or_injection_non_insulin_dm_meds": "dm_noninsulin",
    "starting_new_dm_medications_after_nht_start": "dm_meds_post",
    "a1c_measured_3_months_before_or_3_months_after_starting_nht": "a1c_checked",
    "blood_sugar_>200_before_or_after_the_first_3_months_of_nht": "glucose_over_200",
    "on_asa_before_or_after_the_first_3_months_of_nht": "asa_use",
    "saw_cards_prior_to_nht_start_within_three_months": "cards_prior",
    "saw_cards_after_nht_start": "cards_post",
    "referred_to_cards_post_nht_start_by_oncology": "cards_referral",
    "documented_counseling_about_diet_by_oncology_w_i_3_mos_of_nht_start": "diet_counseling",
    "documented_counseling_about_exercise_by_oncology_w_i_3_mos_of_nht_start": "exercise_counseling",
    "order_for_echo_w_i_3_mos_of_nht_start": "echo_ordered",
    "ecg_w_i_3_month_of_nht_start": "ecg_done",
    "actively_seen_by_non_oncology_providers": "non_onc_provider",
    "has_pcp_.1": "has_pcp_duplicate"}

new_cols = [clean_column_name(c) for c in old_cols]

counts = {}
unique_cols = []

for col in new_cols:
    if col in counts:
        counts[col] += 1
        unique_cols.append(f"{col}_{counts[col]}")
    else:
        counts[col] = 0
        unique_cols.append(col)

df.columns = unique_cols

df = df.rename(columns=MANUAL_RENAME)

## Type Conversion
- all numeric columns -> floats
- date columns -> datetime
- else -> string

In [145]:
#float columns
numeric_float_cols = ["unique_patient_id", "ethnicity", "bmi", "age", "hx_smoking", "family_hx_hd", "hx_htn", "sbp", "dbp", "bp_meds_prior", "bb_prior", "ace_arb_prior", "bp_meds_post", "has_pcp", "hx_hld", "hx_high_tg", "statin_prior", "other_lipid_prior", "lipid_meds_post", "lipid_panel_checked", "tc", "hdl", "ldl", "ascvd_10yr", "ascvd_10yr_pct", "hx_cad", "hx_mi_stent", "hx_chf", "ef", "hx_arrhythmia", "hx_carotid", "hx_pad", "hx_cva", "hx_dm2", "dm_noninsulin", "on_insulin", "dm_meds_post", "a1c_checked", "a1c", "glucose_over_200", "asa_use", "cards_prior", "cards_post", "cards_referral", "diet_counseling", "exercise_counseling", "echo_ordered", "ecg_done", "qtc", "non_onc_provider"]

for col in numeric_float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

#str columns
string_cols = ["specific_nht_used", "adt_agent", "prescribing_provider"]

for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": None})

#date columns
date_cols = [c for c in df.columns if "date" in c]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Dictionary counting datatypes
dtype_counts = df.dtypes.value_counts().to_dict()
print(dtype_counts)

{dtype('float64'): 51, dtype('<M8[ns]'): 10, dtype('O'): 3}


## Save processed data

In [146]:
df.to_csv(PROCESSED_FILE, index=False)
print(f"Renamed dataset saved to: {PROCESSED_FILE}")
df.dtypes

Renamed dataset saved to: ../data/processed/cardio_onc_prostate_02processed.csv


unique_patient_id              float64
ethnicity                      float64
nht_auth_date           datetime64[ns]
nht_start_date          datetime64[ns]
bmi                            float64
                             ...      
ecg_done                       float64
qtc                            float64
non_onc_provider               float64
has_pcp_duplicate              float64
prescribing_provider            object
Length: 64, dtype: object

## Build Column Key

In [147]:
manual_descriptions = {
    "unique_patient_id": "(int) De-identified unique patient identifier",
    "ethnicity": "(int) Patient-reported ethnicity (1- Caucasian; 2- Hispanic; 3- Asian; 4- Black; 5- other)",
    "nht_auth_date": "(date) Date of prior authorization approval for novel hormonal therapy (NHT)",
    "nht_start_date": "(date) Date NHT was initiated",
    "bmi": "(float) Body mass index at time of NHT initiation",
    "specific_nht_used": "(str) Type of novel hormonal therapy prescribed (Abiraterone, Enzalutamide, Apalutamide, Darolutamide)",
    "age": "(int) Age at NHT initiation",
    "adt_start_date": "(date) Date androgen deprivation therapy (ADT) was initiated",
    "adt_agent": "(str) Specific ADT agent prescribed (Lupron, Firmagon, Orgovyx, or Bicalutamide)",
    "hx_smoking": "(int) History of tobacco use prior to NHT start (0- No; 1- Former, quit <1 yr; 2- Active)",
    "family_hx_hd": "(int) Family history of heart disease (1-Yes; 2-No; 3-Not recorded)",
    "hx_htn": "(int) History of hypertension prior to NHT start (1- Yes; 2- No)",
    "sbp": "(int) Systolic blood pressure (mmHg)",
    "dbp": "(int) Diastolic blood pressure (mmHg)",
    "bp_meds_prior": "(int) BP medications within 3 months before NHT (0: None; 1: One; 2: More than two)",
    "bb_prior": "(int) Beta-blocker use prior to NHT (1- Yes; 2- No)",
    "ace_arb_prior": "(int) ACE inhibitor or ARB use prior to NHT (1- Yes; 2- No)",
    "bp_meds_post": "(int) New BP medication after NHT start (0: None; 1: One; 2: More than two)",
    "has_pcp": "(int) Primary care provider status (1: No; 2: UCI; 3: Outside)",
    "hx_hld": "(int) Hyperlipidemia history (1- Yes; 2- No)",
    "hx_high_tg": "(int) Elevated triglycerides history (1- Yes; 2- No)",
    "statin_prior": "(int) Statin use prior to NHT (1- Yes; 2- No)",
    "other_lipid_prior": "(int) Non-statin lipid medication prior to NHT (1- Yes; 2- No)",
    "lipid_meds_post": "(int) New lipid medication after NHT (0: None; 1: One; 2: More than two)",
    "lipid_panel_checked": "(int) Lipid panel within 3 months before/after NHT (1- Yes; 2- No)",
    "tc": "(float) Total cholesterol (mg/dL)",
    "hdl": "(float) HDL cholesterol (mg/dL)",
    "ldl": "(float) LDL cholesterol (mg/dL)",
    "ascvd_10yr": "(float) 10-year ASCVD risk score",
    "ascvd_10yr_pct": "(float) 10-year ASCVD risk percentage",
    "hx_cad": "(int) Coronary artery disease history (1- Yes; 2- No)",
    "cad_date": "(date) Date of CAD diagnosis",
    "hx_mi_stent": "(int) History of MI or stent (1- Yes; 2- No)",
    "mi_stent_date": "(date) Date of MI or stent placement",
    "hx_chf": "(int) Heart failure history (1- Yes; 2- No)",
    "hf_date": "(date) Date of heart failure diagnosis",
    "ef": "(float) Ejection fraction (%)",
    "hx_arrhythmia": "(int) Arrhythmia history (1- Yes; 2- No)",
    "arrhythmia_date": "(date) Date of arrhythmia diagnosis",
    "hx_carotid": "(int) Carotid artery disease history (1- Yes; 2- No)",
    "carotid_date": "(date) Date of carotid disease diagnosis",
    "hx_pad": "(int) Peripheral arterial disease history (1- Yes; 2- No)",
    "pad_date": "(date) Date of PAD diagnosis",
    "hx_cva": "(int) Stroke history (1- Yes; 2- No)",
    "cva_date": "(date) Date of stroke",
    "hx_dm2": "(int) Type II diabetes history (1- Yes; 2- No)",
    "dm_noninsulin": "(int) Non-insulin DM meds (0: None; 1: One; 2: More than one)",
    "on_insulin": "(int) Insulin use (1- Yes; 2- No)",
    "dm_meds_post": "(int) New diabetes meds after NHT (0: None; 1: One; 2: More than one)",
    "a1c_checked": "(int) A1C measured within 3 months (1- Yes; 2- No)",
    "a1c": "(float) Hemoglobin A1c value",
    "glucose_over_200": "(int) Blood glucose >200 mg/dL (1- Yes; 2- No)",
    "asa_use": "(int) Aspirin use (1- Yes; 2- No)",
    "cards_prior": "(int) Cardiology visit within 3 months before NHT (1- Yes; 2- No)",
    "cards_post": "(int) Cardiology visit after NHT (1- Yes; 2- No)",
    "cards_referral": "(int) Cardiology referral from oncology (1- Yes; 2- No)",
    "diet_counseling": "(int) Dietary counseling documented (1- Yes; 2- No)",
    "exercise_counseling": "(int) Exercise counseling documented (1- Yes; 2- No)",
    "echo_ordered": "(int) Echocardiogram ordered (1- Yes; 2- No)",
    "ecg_done": "(int) ECG performed (1- Yes; 2- No)",
    "qtc": "(float) Corrected QT interval (ms)",
    "non_onc_provider": "(int) Followed by non-oncology provider (0- No; 1- Cardio; 2- PCP/Geri; 3- Endo)",
    "prescribing_provider": "(str) Provider who prescribed NHT"
}

column_key = pd.DataFrame({
    "old_name": old_cols,
    "new_name": df.columns,
    "description": [manual_descriptions.get(col, "") for col in df.columns],
})

column_key.to_csv(COLUMN_KEY_FILE, index=False)
print(f"Column key saved to: {COLUMN_KEY_FILE}")

Column key saved to: ../data/column_key.csv


## Create Lookup Fx
- input processed column name & get column description & original name
    -  if no input returns all column descriptions

In [148]:
def describe_column(new_name=None):
    match = column_key[column_key["new_name"] == new_name]

    if new_name is None or match.empty:
        if new_name is not None:
            print(f"Column '{new_name}' not found.")
        for _, row in column_key.iterrows():
            print("New Name:      ", row["new_name"])
            print("Original Name: ", row["old_name"])
            print("Description:   ", row["description"])
            print("-" * 60)
        return

    row = match.iloc[0]
    print("New Name:      ", row["new_name"])
    print("Original Name: ", row["old_name"])
    print("Description:   ", row["description"])


In [149]:
describe_column()

New Name:       unique_patient_id
Original Name:  Unique Patient ID
Description:    (int) De-identified unique patient identifier
------------------------------------------------------------
New Name:       ethnicity
Original Name:  Ethnicity
Description:    (int) Patient-reported ethnicity (1- Caucasian; 2- Hispanic; 3- Asian; 4- Black; 5- other)
------------------------------------------------------------
New Name:       nht_auth_date
Original Name:  NHT Prior authorization date (Abiraterone, Enzalutamide, Apalutamide, Darolutamide)
Description:    (date) Date of prior authorization approval for novel hormonal therapy (NHT)
------------------------------------------------------------
New Name:       nht_start_date
Original Name:  Start date of NHT (Abiraterone, Enzalutamide, Apalutamide, Darolutamide)
Description:    (date) Date NHT was initiated
------------------------------------------------------------
New Name:       bmi
Original Name:  BMI (when they started NHT) 
Description: 